# Numerical Linear Algebra

In [90]:
from numpy.typing import NDArray
import numpy as np
import numpy.linalg as la

np.set_printoptions(precision=15, suppress=False)

## Norms

We know that $Ax=b$ has a unique solution for any nonsingular $n\times n$ matrix $A$ and vector $x$ of size $n$. It follows that if we compute $b=Ax$ and then solve the equation $A\hat{x}=b$, we should find that $x=\hat{x}$. Indeed, this is true mathematically, but computationally, this is not always the case.

Let's test our computer's ability to solve $Ux=b$ for a random upper triangular matrix $U$.

In [91]:
def upper_triangular_solve(n: int) -> NDArray[np.float64]:
    rng = np.random.default_rng(0)

    U = np.triu(rng.random((n, n)))
    x = rng.random((n, 1))

    b = U @ x
    xhat = la.solve(U, b)

    return xhat - x, b - U @ xhat, U

In [92]:
def test_upper_triangular_solve(n: int) -> bool:
    solution, _, _ = upper_triangular_solve(n)
    return np.array_equal(solution, np.zeros((n, 1)))

In [93]:
test_upper_triangular_solve(3)

True

As we can see, our computer solved this problem perfectly! Let's try again, but this time using a larger matrix.

In [94]:
test_upper_triangular_solve(100)

False

This time, the computer got the incorrect solution. What went wrong? Let's try using norms to measure the Euclidean lengths of $\hat{x}-x$ and $b-Ux$.

This should give us some insight into the source of the error.

---



In [95]:
la.norm(upper_triangular_solve(100)[0])

np.float64(415.56990748283164)

In [96]:
la.norm(upper_triangular_solve(100)[1])

np.float64(2.6911403062632485e-14)

The norm of $\hat{x}-x$ is huge, but the norm of $b-Ux$ is tiny! It seems that we computed the wrong answer, but that $\hat{x}$ is a solution to an only slightly changed problem.

Recall that an upper triangular matrix is singular if and only if there is a zero on its diagonal. Let's take a look at the nearest element to zero on the diagonal of our $3\times 3$ and $100\times 100$ matrices.

In [98]:
min(abs(np.diag(upper_triangular_solve(3)[2])))

np.float64(0.5436249914654229)

In [99]:
min(abs(np.diag(upper_triangular_solve(100)[2])))

np.float64(0.013949826634577556)

The reason for our problems in the $100\times 100$ case seems to be that $U$ is nearly singular. The near-zero value along the diagonal seems to have caused floating point errors.

We will now introduce the concept of the *condition number* of a matrix, a measure of how sensitive the matrix is to changes in the input.

Formally, we define the condition number of a matrix $A$ as

$$\kappa(A)=\|A\|\|A^{-1}\|.$$

The condition number has the property:

$$\frac{\|\hat{x}-x\|}{\|x\|}\leq\kappa(A)\frac{\|b-A\hat{x}\|}{\|b\|}.$$

The condition number of any matrix is always at least 1. We call matrices with small condition numbers *well-conditioned* and matrices with large condition numbers *ill-conditioned*.

Now let's take a look of the condition number of our matrices.

In [101]:
la.cond(upper_triangular_solve(3)[2])

np.float64(4.073415412028084)

In [100]:
la.cond(upper_triangular_solve(100)[2])

np.float64(8.078607881200349e+18)

As we can see, the condition number of the $3\times 3$ matrix is small. A perturbation in $b$ can be amplified by at most $\approx 4.07$ in the solution $x$.

The condition number for the $100\times 100$ matrix is massive. A perturbation in $b$ can be amplified by up to $\approx 18.08\times10^{18}$ (18 quintillion!) in the solution $x$.